## CIND820 Baseline Modeling

In [51]:
import pandas as pd

model_prep = pd.read_csv('2026-27 QMJHL PBP prepared.csv')

### Final preprocessing steps

#### One-hot encoding

To one-hot encode the 'strength' categorical variable, we'll use OneHotEncoder from the sklearn.preprocessing package. This will create a binary variable for every strength in the dataset, like 'strength_5v5' or 'strength_3v3.' This follows how Younggren & Younggren encoded strength in their 2021 NHL expected goals model.

In [52]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
encoder.set_output(transform='pandas')

strength_encoded = encoder.fit_transform(model_prep[['strength']])
model_prep = pd.concat([model_prep, strength_encoded], axis=1)

To one-hot encode the 'score_diff' variable, extreme scores (up/down 4 or more) will be combined into a singular variable because of the rareness of the very high/low values, like +/-11. These are the thresholds used by Younggren & Younggren in their 2021 development of an NHL expected goals model.

In [53]:
import numpy as np

model_prep['score_even'] = np.where(model_prep['score_diff'] == 0, 1, 0)
model_prep['score_up_1'] = np.where(model_prep['score_diff'] == 1, 1, 0)
model_prep['score_down_1'] = np.where(model_prep['score_diff'] == -1, 1, 0)
model_prep['score_up_2'] = np.where(model_prep['score_diff'] == 2, 1, 0)
model_prep['score_down_2'] = np.where(model_prep['score_diff'] == -2, 1, 0)
model_prep['score_up_3'] = np.where(model_prep['score_diff'] == 3, 1, 0)
model_prep['score_down_3'] = np.where(model_prep['score_diff'] == -3, 1, 0)
model_prep['score_up_4'] = np.where(model_prep['score_diff'] >= 4, 1, 0)
model_prep['score_down_4'] = np.where(model_prep['score_diff'] <= -4, 1, 0)

#### Transforming the target variable

The target variable in these models will be named 'is_goal,' where 1 = a goal and 0 = not a goal.

In [54]:
model_prep['is_goal'] = np.where(model_prep['event'] == 'goal', 1, 0)

#### Variable selection

The code below transforms the model_prep DataFrame to include only the information and variables needed for the models. These variables are very similar to those outlined in Younggren & Younggren's NHL expected goals model.

In [55]:
model_prep = model_prep[['game_id',
 'player_team_id', 'player',
 'goalie_team_id', 'goalie',
 'shot_distance',
 'shot_angle',
 's',
 'period_id',
 'x_location', 'y_location',
 'x_prev', 'y_prev',
 'distance_prev',
 's_since_prev',
 'home',
 'strength',
 'strength_3v3', 'strength_3v4', 'strength_3v5', 'strength_4v3', 'strength_4v4', 'strength_4v5', 'strength_5v3', 'strength_5v4', 'strength_5v5',
 'score_diff',
 'score_even', 'score_up_1', 'score_down_1', 'score_up_2', 'score_down_2', 'score_up_3', 'score_down_3', 'score_up_4', 'score_down_4',
 'is_goal']]

#### Split train and test sets

The train set will be all games before October 19th, 2026 (100 games). The game_id of this final game before the cutoff was 33205. The code below returns the last chronological row in this game, of which its index will be where we split the data.

In [56]:
model_prep[model_prep['game_id'] == 33205].iloc[-1:]

,game_id,player_team_id,player,goalie_team_id,goalie,shot_distance,shot_angle,s,period_id,x_location,...,score_even,score_up_1,score_down_1,score_up_2,score_down_2,score_up_3,score_down_3,score_up_4,score_down_4,is_goal
5814,33205,Quebec,Charles-Antoine Dubé,Victoriaville,Anthony Catanzariti,11.28401,23.687526,3600,3,21.333333,...,0,0,0,0,0,0,0,1,0,0


We will copy the model_prep DataFrame, containing only the variables and information needed in the model, to create new DataFrames accurately labeled as model_train and model_test.

In [58]:
model_train = model_prep.loc[:5814].copy()
model_test = model_prep.loc[5815:].copy().reset_index()

## Logistic regression

#### Data transformation: replacing NaN values

In sklearn's linear_model package, the LogisticRegression class does not natively accept NaN values. These occur in our dataset when it comes to the first shot in every period, as 'x_prev,' 'y_prev,' 'distance_prev,' and 's_since_prev' are left blank because there was no shot preceding them. To provide a solution for this model, we will consider the faceoff at the beginning of the period as our previous event. This means that 'x_prev' and 'y_prev' will be 100 and 42.5 respectively (the location of the faceoff dot). The previous time in 's_since_prev' will be the beginning of the period.

For example, the very first row in our dataset has these values:
* 's' = 86
* 'x_location' = 41
* 'y_location' = 21.5333333333333

In this case, the NaN rows will be populated:
* 's_since_prev' = 86
* 'x_prev' = 100
* 'y_prev' = 42.5
* 'distance_prev' = 62.6147

In [59]:
lr_prep = model_prep.copy()

lr_prep['x_prev'] = lr_prep['x_prev'].fillna(100)
lr_prep['y_prev'] = lr_prep['y_prev'].fillna(42.5)

lr_prep['s_since_prev'] = np.where((lr_prep['s_since_prev'].isna()) & (lr_prep['period_id'] == 1), lr_prep['s'], lr_prep['s_since_prev'])
lr_prep['s_since_prev'] = np.where((lr_prep['s_since_prev'].isna()) & (lr_prep['period_id'] == 2), (lr_prep['s'] - 1200), lr_prep['s_since_prev'])
lr_prep['s_since_prev'] = np.where((lr_prep['s_since_prev'].isna()) & (lr_prep['period_id'] == 3), (lr_prep['s'] - 2400), lr_prep['s_since_prev'])
lr_prep['s_since_prev'] = np.where((lr_prep['s_since_prev'].isna()) & (lr_prep['period_id'] == 4), (lr_prep['s'] - 3600), lr_prep['s_since_prev'])

lr_prep['distance_prev'] = np.sqrt((lr_prep['x_location'] - lr_prep['x_prev']) ** 2 + (lr_prep['y_location'] - lr_prep['y_prev']) ** 2)

#### Data grouping: strength situations

Younggren & Younggren's NHL expected goals model makes the conscious choice to divide itself into three separate models based on 'strength' - we will call these three situations
* even strength (EV): when both teams have the same number of players on the ice (5v5, 4v4, 3v3)
* power play (PP): when the shooting team has more players on the ice than the other team because of a penalty (5v4, 5v3, 4v3)
* penalty kill (PK): when the shooting team has fewer players on the ice than the other team because of a penalty (4v5, 3v5, 3v4)

In [133]:
even_cols = ['strength_5v5', 'strength_4v4', 'strength_3v3']
lr_even = lr_prep[lr_prep[even_cols].eq(1).any(axis=1)]
pp_cols = ['strength_5v4', 'strength_5v3', 'strength_4v3']
lr_pp = lr_prep[lr_prep[pp_cols].eq(1).any(axis=1)]
pk_cols = ['strength_4v5', 'strength_3v5', 'strength_3v4']
lr_pk = lr_prep[lr_prep[pk_cols].eq(1).any(axis=1)]

### Even strength (EV) logistic regression model

First, we will do the even strength logistic regression model since this is the 'default' game state and this state makes up the majority of playing time in most hockey games. From the features, we removed the Game ID, the target variable ('is_goal'), text variables (teams and players), as well as the 'strength' and 'score_diff' variables that our encoded variables were derived from. We are also dropping 'period_id' because it is already reflected with greater accuracy in the 's' (time elapsed in game) variable. Younggren & Younggren's analysis similarly drops the period as a numeric variable, which is why this decision was made.

In [134]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, log_loss

features = lr_even.drop(columns=['game_id', 'is_goal', 'player_team_id', 'player', 'goalie_team_id', 'goalie', 'strength', 'score_diff', 'period_id'])
targets = lr_even['is_goal']

X_train = lr_even.loc[:5814].drop(columns=['game_id', 'is_goal', 'player_team_id', 'player', 'goalie_team_id', 'goalie', 'strength', 'score_diff', 'period_id'])
y_train = lr_even.loc[:5814, 'is_goal']

X_test = lr_even.loc[5815:].drop(columns=['game_id', 'is_goal', 'player_team_id', 'player', 'goalie_team_id', 'goalie', 'strength', 'score_diff', 'period_id'])
y_test = lr_even.loc[5815:, 'is_goal']

model_lr = LogisticRegression(max_iter=10000)
model_lr.fit(X_train, y_train)

,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",10000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 

The three evaluation metrics below were all used previously in the literature to evaluate NHL expected goals models - both AUC and log loss were displayed in Younggren & Younggren's 2021 model, while RMSE was mentioned in Sprigings & Toumi's 2015 work. All of these metrics for this model score around the 'fair/acceptable' range by these metrics.

In [136]:
ytestpred = model_lr.predict_proba(X_test)[:, 1]

roc_auc = roc_auc_score(y_test, ytestpred)
print(f"ROC-AUC: {roc_auc:.3f}")

loss = log_loss(y_test, ytestpred)
print(f"Log loss: {loss:.4f}")

rmse = np.sqrt(np.mean((ytestpred - y_test) ** 2))
print(f"RMSE: {rmse:.4f}")

ROC-AUC: 0.738
Log loss: 0.2873
RMSE: 0.2874


Below are the coefficients of each feature in the model, listed in absolute order of magnitude from highest to lowest. The top of the list is dominated by the encoded strengths and score differentials, while shot distance is the strongest coefficient that does not relate to those two dimensions. Provided how Younggren & Younggren's model was dominated by more 'immediate' shot factors like shot distance and seconds since previous shot, it may be useful to train another logistic regression model on a set of features that excludes the strength and score differential features.

It would be interesting to compare this model to the more expansive model above, to see whether a shot's immediate circumstances (such as its location and its timing) are better predictors than the existing state of the game that cannot be changed by a single shot. The second strongest coefficient is 'score_down_4,' and its negative value means that the expected goals on a shot by a team that is down by four or more goals is greatly diminished by this circumstance. I believe that this shows up so extremely because teams that are losing by four or more goals are often vastly outmatched by their opponents and struggle greatly to score.

These score effects are perhaps much more emphasized in the QMJHL as compared to the NHL because poor-performing teams are seen as part of the "junior hockey cycle" - because players can only play a maximum of five years in the league, there is a lot of turnover year-by-year and teams tend to "accept" poor-performing seasons in order to get better draft picks and select players who will help the team win in 2-4 years. Having these score differentials in the logistic regression model may blur the lines and place an outsized importance on the teams that players are on, rather than evaluating the true quality of their shot.

This could cause considerable problems when answering Question 2, where we want to find the top xG generators on poor-performing teams. This could cause us to overlook a player who is on a team that loses by four or more goals often, where their xG is greatly diminished by their team's situation.

In [135]:
coef_table = pd.DataFrame({
    'Feature': features.columns,
    'Coefficient': model_lr.coef_[0]
})

coef_table = coef_table.sort_values(by='Coefficient', key=abs, ascending=False)

coef_table

,Feature,Coefficient
18,strength_5v5,-0.481756
27,score_down_4,-0.451805
14,strength_4v4,-0.254622
21,score_down_1,-0.227530
22,score_up_2,-0.178778
25,score_down_3,0.166019
20,score_up_1,-0.160218
0,shot_distance,-0.147693
9,home,0.126280
10,strength_3v3,-0.118994


The code below adds a variable named 'xg_lr' to the lr_even DataFrame. This variable contains the predicted probabilities from the logistic regression model. This is what expected goals are - a single shot's probability of becoming a goal, on a scale from 0 to 1.

In [138]:
lr_even['xg_lr'] = model_lr.predict_proba(features)[:, 1]

/var/folders/0n/31k33nxn59sgvdqjp51nfl2c0000gn/T/ipykernel_3295/559152845.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  lr_even['xg_lr'] = model_lr.predict_proba(features)[:, 1]


Below are the top ten shots by xG in the training set. We can see that the maximum xG value is ~0.38. These values are relatively low because the approximate naive probability of a shot becoming a goal is 10% (assuming a .900 average save percentage across the league). Of these top ten 'most valuable' shots in the training set, three of them were goals.

In [141]:
lr_even[['strength', 'player_team_id', 'player', 'score_diff', 'goalie_team_id', 'shot_distance', 'shot_angle', 's_since_prev', 'is_goal', 'xg_lr']].loc[:5814].sort_values(by='xg_lr', ascending=False).head(10)

,strength,player_team_id,player,score_diff,goalie_team_id,shot_distance,shot_angle,s_since_prev,is_goal,xg_lr
4060,4v4,Newfoundland,Benjamin Veitch,-2,Rimouski,6.557523,15.025222,104.0,0,0.375929
5595,5v5,Rimouski,Jérémy Turcot,-3,Chicoutimi,5.916174,16.699244,16.0,0,0.364095
4254,3v3,Halifax,Shawn Carrier,0,Drummondville,9.118053,18.103763,27.0,0,0.357212
3820,5v5,Gatineau,Gabriel Séguin,-3,Baie-Comeau,8.740455,7.450249,21.0,1,0.346355
1661,5v5,Rimouski,Rafaël Cloutier,10,Shawinigan,8.045030,6.064922,110.0,1,0.344367
5443,5v5,Shawinigan,Gleb Olkhovik,4,Charlottetown,7.666667,0.000000,105.0,1,0.344218
3556,5v5,Charlottetown,Cameron Haye,0,Drummondville,6.006686,2.703626,194.0,0,0.342134
3555,4v4,Charlottetown,Kage Flory,0,Drummondville,6.762314,9.648045,56.0,0,0.340974
3581,5v5,Drummondville,Yoan Tassé,-2,Charlottetown,5.071735,9.648045,2.0,0,0.339533
123,3v3,Cape Breton,Noah Jettelson,0,Halifax,9.769923,8.337444,44.0,0,0.332645


Below are the top ten shots by xG in the test set, as predicted by the logistic regression model. The maximum xG value here is also ~0.38. Interesting, three of these top ten shots were also goals.

In [115]:
lr_even[['strength', 'player_team_id', 'player', 'score_diff', 'goalie_team_id', 'shot_distance', 'shot_angle', 's_since_prev', 'is_goal', 'xg_lr']].loc[5815:].sort_values(by='xg_lr', ascending=False).head(10)

,strength,player_team_id,player,score_diff,goalie_team_id,shot_distance,shot_angle,s_since_prev,is_goal,xg_lr
8022,5v5,Sherbrooke,Thomas Rousseau,-3,Halifax,5.730062,8.530766,12.0,1,0.379583
8207,3v3,Moncton,Liam Kilfoil,0,Newfoundland,8.045030,6.064922,126.0,0,0.376683
6534,5v5,Halifax,Mateo Nobert,4,Charlottetown,6.672685,2.433606,145.0,0,0.359747
8905,5v5,Charlottetown,Matthew Butler,0,Cape Breton,5.400643,9.055358,304.0,0,0.344871
6540,5v5,Halifax,Oleg Kulebiakin,4,Charlottetown,9.703965,5.025152,107.0,0,0.332317
9227,5v5,Gatineau,Zakary Horvat-Edouard,-3,Val-d'Or,6.880003,14.305552,2.0,1,0.330424
8873,5v5,Halifax,Patrick McNab,-2,Saint John,8.376571,5.824028,6.0,0,0.329023
7118,5v5,Newfoundland,Benjamin Veitch,-2,Charlottetown,10.700480,4.556130,158.0,1,0.326429
6614,3v3,Newfoundland,Louis-François Bélanger,0,Victoriaville,11.600299,18.513044,54.0,0,0.324880
7969,5v5,Quebec,Mavrick Rousseau-Hamel,-2,Baie-Comeau,7.041464,18.778033,39.0,0,0.323104


In the next steps, I will explore other models (such as XGBoost) and also build a version of the best-performing model without the strength and score differential attributes, primarily to answer Question 2 (evaluating players' individual performances).